## ChatMemory
Abstraction helps us maintain list of `ChatMessage`s. It has features to:
- evict older messages
- summarise multiple or one message
- remove unimportant information
- add additional information (RAG)

## Eviction Policy
Since most LLMs have a cap on how many tokens to process at a time, as the conversation progresses, we should be able to evict some messages. It also helps control cost and time taken to process a request.

**Context Window:** is the amount of information an LLM can use at a time while generating response. When we prompt an LLM, for example ChatGPT, it doesn't store all chat history. For every message, it receives a block called context which contains:
- system instructions
- chat history
- uploaded files
- retreived documents (RAG)
- tool output

Different models have different context size limits ranging from 8k to 1M tokens.

## Persistence
`ChatMemoryStore` instance helps us persist chat memory (rather than keeping it in RAM which is the default).

In [ ]:
public interface ChatMemoryStore {

    // Retrieves messages for a specified chat memory.
    List<ChatMessage> getMessages(Object memoryId);

    // Updates messages for a specified chat memory.
    // Called everytime a new message is added to ChatMemory
    void updateMessages(Object memoryId, List<ChatMessage> messages);

    // Deletes all messages for a specified chat memory.
    void deleteMessages(Object memoryId);
}

A sample implementation that stores chat memory in Postgres. The table schema can look like:
```sql
CREATE TABLE chat_messages (
    id BIGSERIAL PRIMARY KEY,
    memory_id VARCHAR(100),
    role VARCHAR(20),
    content TEXT,
    created_at TIMESTAMP
);
```

In [ ]:
public class PostgresChatMemoryStore implements ChatMemoryStore {
    private final Connection conn;
    public PostgresChatMemoryStore(Connection conn) {
        this.conn = conn;
    }

    @Override
    public List<ChatMessage> getMessages(Object memoryId) {
        List<ChatMessage> messages = new ArrayList<>();

        String sql = """
                    SELECT role, content
                    FROM chat_messages
                    WHERE memory_id = ?
                    ORDER BY id
                """;

        try {
            PreparedStatement ps = conn.prepareStatement(sql);
            ps.setString(1, memoryId.toString());

            ResultSet rs = ps.executeQuery();
            while (rs.next()) {
                String role = rs.getString("role");
                String content = rs.getString("content");

                switch (role) {
                    case "user" -> messages.add(UserMessage.from(content));
                    case "ai" -> messages.add(AiMessage.from(content));
                    case "system" -> messages.add(SystemMessage.from(content));
                }
            }
        } catch (SQLException e) {
            throw new RuntimeException(e);
        }

        return messages;
    }

    @Override
    public void updateMessages(Object memoryId, List<ChatMessage> messages) {
        deleteMessages(memoryId);

        String sql = """
                    INSERT INTO chat_messages
                    (memory_id, role, content)
                    VALUES (?, ?, ?)
                """;

        try {
            PreparedStatement ps = conn.prepareStatement(sql);

            for (ChatMessage message : messages) {
                ps.setString(1, memoryId.toString());

                if (message instanceof UserMessage um) {
                    ps.setString(2, "user");
                    ps.setString(3, um.singleText());
                } else if (message instanceof AiMessage am) {
                    ps.setString(2, "ai");
                    ps.setString(3, am.text());
                } else if (message instanceof SystemMessage sm) {
                    ps.setString(2, "system");
                    ps.setString(3, sm.text());
                }

                ps.addBatch();
            }

            ps.executeBatch();
        } catch (SQLException e) {
            throw new RuntimeException(e);
        }
    }

    @Override
    public void deleteMessages(Object memoryId) {

        String sql = """
                    DELETE FROM chat_messages
                    WHERE memory_id = ?
                """;

        try {
            PreparedStatement ps = conn.prepareStatement(sql);
            ps.setString(1, memoryId.toString());
            ps.executeUpdate();
        } catch (SQLException e) {
            throw new RuntimeException(e);
        }
    }
}

## Implementation
LangChain4J provides two built-in implementation of `ChatMemory`:
- `MessageWindowChatMemory` : sliding window, retains the N most recent messages and evicts older ones that no longer fit. Keep in mind that each message can have varying number of tokens, therefore this implementation may not be the best.
- `TokenWindowChatMemory`: sliding window but focuses on keeping the N most recent tokens. Requires `TokenCountEstimator`.

In [ ]:
ChatMemory messageWindowChatMemory = MessageWindowChatMemory.builder()
    .id("random-chat-id")
    .alwaysKeepSystemMessageFirst(true)
    .maxMessages(4)
    .build();

/*
1 User: Hello
2 AI: Hi
3 User: Explain Kafka
4 AI: Kafka is...
5 User: Show examples
6 AI: ...
---- transformed to ----
1 User: Explain Kafka
2 AI: Kafka is...
3 User: Show examples
4 AI: ...

It is easy to exceed limit since messages can have wildly different sizes
1 AI: Hi
2 User: <1000 word prompt>
*/

In [ ]:
ChatMemory tokenWindowChatMemory = TokenWindowChatMemory.builder()
    .id("random-chat-id-2")
    .alwaysKeepSystemMessageFirst(true)
    .maxTokens(6000, GoogleAiGeminiTokenCountEstimator.builder()
            .modelName("gemini-2.5-flash")
            .build())
    .build();

/*
1 User: hello	5 tokens
2 AI: response	20 tokens
3 User: Large code paste	5000 tokens
4 AI: explanation	2000 tokens
---- total tokens = 7025 ----
Older messages removed till total tokens <= 6000
*/

**System Message:** is always retained and is limited to one. If system message with same content is added, it is retained. If the content varies then it is replaced. `ChatMemory` provides an option to keep system message as the very first message.